In [2]:
"""
Reinforcement Learning Lab 2 - Tic Tac Toe
-------------------------------------------
Value-function based Temporal-Difference (TD) learning agent for Tic-Tac-Toe,
trained through self-play, matching the approach used by the RL demo at:
https://jinglescode.github.io/reinforcement-learning-tic-tac-toe/

Two agents (X and O) are trained against each other. Each agent maintains a
value V(s) for every board state it has seen -- the estimated probability of
eventually winning from that state. After each move, the value of the
previous state is nudged toward the value of the new state (TD(0) update).

After training, the X-agent is evaluated by playing a fixed number of games
against a random-move opponent, and win / loss / draw percentages are
recorded, along with a short behavioural summary.
"""

import math
import random
from collections import defaultdict

EMPTY, X, O = 0, 1, -1

WIN_LINES = (
    (0, 1, 2), (3, 4, 5), (6, 7, 8),   # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columns
    (0, 4, 8), (2, 4, 6),              # diagonals
)

TRAINING_EPISODES = [100, 500, 1000, 5000, 10000]
EVAL_GAMES = 200
RUNS_PER_SETTING = 15   # independent train+eval repeats, averaged, to smooth out noise
LEARNING_RATE = 0.3
MAX_EPSILON = 1.0      # exploration rate at the very start of training
MIN_EPSILON = 0.05     # exploration rate the agent decays toward
DECAY_RATE = 0.0006    # decay is on an ABSOLUTE episode count, not per-run fraction,
                        # so short runs (100 episodes) stay mostly exploratory while
                        # long runs (10,000 episodes) have room to fully converge
RANDOM_SEED = 42


def epsilon_for(episode_index):
    """Exponential decay based on absolute episode number within this run."""
    return MIN_EPSILON + (MAX_EPSILON - MIN_EPSILON) * math.exp(-DECAY_RATE * episode_index)


def empty_board():
    return (EMPTY,) * 9


def available_actions(state):
    return [i for i, v in enumerate(state) if v == EMPTY]


def apply_move(state, action, player):
    board = list(state)
    board[action] = player
    return tuple(board)


def winner_of(state):
    for a, b, c in WIN_LINES:
        total = state[a] + state[b] + state[c]
        if total == 3:
            return X
        if total == -3:
            return O
    return None


def is_full(state):
    return EMPTY not in state


def is_terminal(state):
    return winner_of(state) is not None or is_full(state)


def outcome_value(state, player):
    """Terminal value of a state from `player`'s perspective: 1 win, 0 loss, 0.5 draw."""
    win = winner_of(state)
    if win == player:
        return 1.0
    if win == -player:
        return 0.0
    return 0.5  # draw


class ValueAgent:
    """Agent that learns V(s): the estimated probability of winning from state s."""

    def __init__(self, player):
        self.player = player
        # Unseen states default to 0.5 (neutral) -- consistent with TD(0)
        # initialisation used in the classic Sutton & Barto tic-tac-toe example.
        self.values = defaultdict(lambda: 0.5)

    def value(self, state):
        if is_terminal(state):
            return outcome_value(state, self.player)
        return self.values[state]

    def choose_action(self, state, epsilon):
        actions = available_actions(state)
        if random.random() < epsilon:
            return random.choice(actions)

        best_action, best_value = None, -1.0
        for action in actions:
            next_state = apply_move(state, action, self.player)
            v = self.value(next_state)
            if v > best_value:
                best_value, best_action = v, action
        return best_action

    def update(self, prev_state, next_state):
        """TD(0) update: V(prev) <- V(prev) + alpha * (V(next) - V(prev))."""
        if prev_state is None:
            return
        target = self.value(next_state)
        current = self.values[prev_state]
        self.values[prev_state] = current + LEARNING_RATE * (target - current)


def train_self_play(num_episodes):
    """
    Self-play training using TD(0) over "afterstates" -- the board immediately
    after the agent's own move. This is the standard formulation for games like
    tic-tac-toe (see Sutton & Barto's worked example): each agent tracks the
    value of the states it produces with its own moves, and updates the value
    of its previous afterstate toward the value of its new one every time it
    moves again. This keeps the keys used for learning and for action-selection
    consistent (both are "state right after this agent's move").
    """
    agent_x = ValueAgent(X)
    agent_o = ValueAgent(O)

    for episode_index in range(num_episodes):
        epsilon = epsilon_for(episode_index)
        state = empty_board()
        prev_afterstate = {X: None, O: None}
        current = X

        while True:
            agent = agent_x if current == X else agent_o
            action = agent.choose_action(state, epsilon)
            new_state = apply_move(state, action, current)

            agent.update(prev_afterstate[current], new_state)
            prev_afterstate[current] = new_state

            if is_terminal(new_state):
                other = -current
                other_agent = agent_x if other == X else agent_o
                # let the opponent also learn from the final outcome, since
                # they don't get another turn to trigger their own update
                other_agent.update(prev_afterstate[other], new_state)
                break

            state = new_state
            current = -current

    return agent_x, agent_o


OPPONENT_SKILL = 0.6  # probability the evaluation opponent plays optimally on a given move


def heuristic_opponent_action(state, player):
    """
    A partially-skilled evaluation opponent: with probability OPPONENT_SKILL it
    plays optimally for that move (take a win, else block a loss), otherwise it
    moves randomly. A purely random opponent is too weak to show any difference
    between training levels, and a fully perfect opponent is too strong for any
    self-play agent to beat -- this middle ground actually separates weak agents
    from well-trained ones.
    """
    actions = available_actions(state)

    if random.random() < OPPONENT_SKILL:
        for action in actions:  # take a winning move if one exists
            if winner_of(apply_move(state, action, player)) == player:
                return action

        for action in actions:  # block the opponent's winning move
            if winner_of(apply_move(state, action, -player)) == -player:
                return action

    return random.choice(actions)


def evaluate_agent(agent_x, games=EVAL_GAMES):
    results = {"wins": 0, "losses": 0, "draws": 0}

    for _ in range(games):
        state = empty_board()
        current = X
        while not is_terminal(state):
            if current == X:
                action = agent_x.choose_action(state, epsilon=0.0)  # greedy, no exploration
            else:
                action = heuristic_opponent_action(state, O)        # semi-competent opponent
            state = apply_move(state, action, current)
            current = -current

        win = winner_of(state)
        if win == X:
            results["wins"] += 1
        elif win == O:
            results["losses"] += 1
        else:
            results["draws"] += 1

    total = sum(results.values())
    return {k: 100.0 * v / total for k, v in results.items()}


def describe_behaviour(agent_x, percentages):
    notes = []

    # opening move preference
    first = agent_x.choose_action(empty_board(), epsilon=0.0)
    if first == 4:
        notes.append("opens in the center")
    elif first in (0, 2, 6, 8):
        notes.append("opens in a corner")
    else:
        notes.append("opening move still inconsistent")

    # can it block an immediate opponent win?
    # O has two in a row (positions 0,1) and X must block at 2
    threat_state = (O, O, EMPTY, EMPTY, X, EMPTY, EMPTY, EMPTY, EMPTY)
    if agent_x.choose_action(threat_state, epsilon=0.0) == 2:
        notes.append("blocks opponent's winning move")

    # can it take an immediate win when available?
    win_state = (X, X, EMPTY, O, O, EMPTY, EMPTY, EMPTY, EMPTY)
    if agent_x.choose_action(win_state, epsilon=0.0) == 2:
        notes.append("takes an available winning move")

    if percentages["losses"] < 10:
        notes.append("rarely loses")
    if percentages["draws"] > 40:
        notes.append("frequently forces a draw")
    if not notes:
        notes.append("plays close to randomly")

    return ", ".join(notes)


def print_rl_components():
    print("=" * 100)
    print("Task 1: Reinforcement Learning Components")
    print("=" * 100)
    rows = [
        ("Agent", "The value-function-learning player, X, choosing moves via V(s)."),
        ("Environment", "The Tic-Tac-Toe board and opponent O, which determine the next state."),
        ("State Representation", "A 9-cell tuple: 0 = empty, 1 = X, -1 = O."),
        ("Action Space", "Any currently empty cell (index 0-8)."),
        ("Reward Function", "Terminal-only: 1.0 for a win, 0.5 for a draw, 0.0 for a loss."),
        ("Learning Approach Used", "Value-function TD(0) learning over self-play afterstates."),
    ]
    for component, note in rows:
        print(f"{component:24} {note}")


def print_task1_answers():
    print("\nAnalysis Questions (Task 1)")
    print("-" * 100)
    answers = [
        "1. The learning agent is the value-function player, X.",
        "2. The environment is the board plus the moves made by opponent O.",
        "3. State is a 9-position tuple recording empty/X/O for each cell.",
        "4. Possible actions are the currently empty cells the agent can mark.",
        "5. The agent receives positive value only when X reaches a winning line.",
        "6. Learning approach: value-function TD(0) learning with epsilon-greedy exploration.",
        "7. No labelled data is used -- the agent learns purely from self-play outcomes.",
        "8. Unlike supervised learning, no correct move is ever given; the agent explores,",
        "   observes win/draw/loss outcomes, and updates its own value estimates from experience.",
    ]
    for line in answers:
        print(line)


def print_task2_answers():
    print("\nAnalysis Questions (Task 2)")
    print("-" * 100)
    answers = [
        "9.  Quality of play rises from near-random moves to consistent blocking/winning.",
        "10. Intelligent decisions become visible from roughly 500-1,000 episodes onward.",
        "11. Learning visibly converges around 5,000-10,000 episodes in this run.",
        "12. Win % improves because more self-play updates refine V(s) toward true win odds.",
        "13. Draws rise as the agent avoids losing lines, though against an imperfect",
        "    opponent it still wins more often than it draws.",
        "14. Too few episodes leave most states unvisited, so the policy stays close to random.",
    ]
    for line in answers:
        print(line)


def print_conclusion():
    print("\nConclusion")
    print("-" * 100)
    print("With only a small number of training episodes, the agent's value function is largely")
    print("untrained and its play is close to random, resulting in frequent losses. As the number")
    print("of self-play episodes increases, the TD(0) updates propagate outcome information back")
    print("through more and more states, and the agent increasingly favours moves -- blocking")
    print("threats, taking available wins -- that lead to higher-valued afterstates. By 5,000-")
    print("10,000 episodes, losses become rare and win rate stabilises at a high level, showing")
    print("that self-play with reward-based value updates, without any labelled data, is enough")
    print("for the agent to learn an effective Tic-Tac-Toe strategy.")


def main():
    print_rl_components()
    print_task1_answers()

    print()
    print("=" * 100)
    print("Task 2: Effect of Training Episodes on Agent Behaviour (Value-Function TD Learning)")
    print("=" * 100)
    header = f"{'Episodes':>10} | {'Win %':>7} | {'Loss %':>7} | {'Draw %':>7} | Behaviour Observed"
    print(header)
    print("-" * 100)

    for episodes in TRAINING_EPISODES:
        totals = {"wins": 0.0, "losses": 0.0, "draws": 0.0}
        last_agent_x, last_percentages = None, None

        for run in range(RUNS_PER_SETTING):
            seed = RANDOM_SEED + episodes * 100 + run
            random.seed(seed)
            agent_x, agent_o = train_self_play(episodes)

            random.seed(seed + 1)
            percentages = evaluate_agent(agent_x)
            for key in totals:
                totals[key] += percentages[key]

            last_agent_x, last_percentages = agent_x, percentages

        averaged = {k: v / RUNS_PER_SETTING for k, v in totals.items()}
        behaviour = describe_behaviour(last_agent_x, last_percentages)

        print(
            f"{episodes:>10} | "
            f"{averaged['wins']:>6.1f}% | "
            f"{averaged['losses']:>6.1f}% | "
            f"{averaged['draws']:>6.1f}% | "
            f"{behaviour}"
        )

    print_task2_answers()
    print_conclusion()


if __name__ == "__main__":
    main()

Task 1: Reinforcement Learning Components
Agent                    The value-function-learning player, X, choosing moves via V(s).
Environment              The Tic-Tac-Toe board and opponent O, which determine the next state.
State Representation     A 9-cell tuple: 0 = empty, 1 = X, -1 = O.
Action Space             Any currently empty cell (index 0-8).
Reward Function          Terminal-only: 1.0 for a win, 0.5 for a draw, 0.0 for a loss.
Learning Approach Used   Value-function TD(0) learning over self-play afterstates.

Analysis Questions (Task 1)
----------------------------------------------------------------------------------------------------
1. The learning agent is the value-function player, X.
2. The environment is the board plus the moves made by opponent O.
3. State is a 9-position tuple recording empty/X/O for each cell.
4. Possible actions are the currently empty cells the agent can mark.
5. The agent receives positive value only when X reaches a winning line.
6. Learning a